# 🚀 Huấn Luyện Mạng Nơ-ron Đồ Thị Không-Thời Gian (ST-GNN) Cho Hợp Nhất Trinh Sát Tàu Biển

Notebook này được tự động xuất từ dự án **Naval ISR Multi-Source Reconnaissance Data Fusion**.
Mục tiêu: Huấn luyện mô hình **SpatioTemporal GNN** trên GPU (Google Colab T4 / A100) để phân loại mục tiêu chiến thuật từ dữ liệu Radar + ESM + AIS + SAR.


In [ ]:
# 1. Kiểm tra môi trường GPU
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# 2. Kiến trúc Mạng Nơ-ron Đồ Thị (ST-GNN)
import torch.nn as nn
import torch.nn.functional as F

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, dropout: float = 0.1):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a_src = nn.Parameter(torch.zeros(size=(out_features, 1)))
        self.a_dst = nn.Parameter(torch.zeros(size=(out_features, 1)))
        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight.data, gain=1.414)
        nn.init.xavier_uniform_(self.a_src.data, gain=1.414)
        nn.init.xavier_uniform_(self.a_dst.data, gain=1.414)

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        Wh = self.W(h)
        f_src = torch.matmul(Wh, self.a_src)
        f_dst = torch.matmul(Wh, self.a_dst)
        e = self.leaky_relu(f_src + f_dst.T)
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=1)
        attention = self.dropout(attention)
        return torch.matmul(attention, Wh)

class SpatioTemporalGNN(nn.Module):
    def __init__(self, in_features: int = 8, hidden_dim: int = 64, num_classes: int = 4, dropout: float = 0.1):
        super().__init__()
        self.input_encoder = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )
        self.gat1 = GraphAttentionLayer(hidden_dim, hidden_dim, dropout=dropout)
        self.gat2 = GraphAttentionLayer(hidden_dim, hidden_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        h0 = self.input_encoder(x)
        h1 = self.gat1(h0, adj)
        h1 = self.norm1(h0 + F.relu(h1))
        h2 = self.gat2(h1, adj)
        h2 = self.norm2(h1 + F.relu(h2))
        return self.classifier(h2)


In [ ]:
# 3. Bộ sinh dữ liệu đồ thị tổng hợp (Synthetic Multi-Sensor Graphs)
def generate_batch_graphs(batch_size: int = 16, num_nodes: int = 12):
    # Node features: [x, y, vx, vy, rcs, esm_freq, ais_flag, length]
    X = torch.randn(batch_size, num_nodes, 8)
    # Adjacency matrix: spatial proximity graph
    adj = torch.randint(0, 2, (batch_size, num_nodes, num_nodes)).float()
    for b in range(batch_size):
        adj[b].fill_diagonal_(1.0)
    # Labels: 0=Destroyer, 1=Frigate, 2=Commercial, 3=SmallCraft
    labels = torch.randint(0, 4, (batch_size, num_nodes))
    return X, adj, labels


In [ ]:
# 4. Vòng lặp Huấn luyện GPU với AdamW và CosineAnnealingLR
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

model = SpatioTemporalGNN(in_features=8, hidden_dim=64, num_classes=4).to(device)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
epochs = 20
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print('Bắt đầu quá trình huấn luyện mô hình trên GPU...')
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for _ in range(50):  # 50 batches per epoch
        X, adj, y = generate_batch_graphs(batch_size=16, num_nodes=10)
        X, adj, y = X.to(device), adj.to(device), y.to(device)

        optimizer.zero_grad()
        # Flatten for classification loss
        batch_logits = []
        for b in range(16):
            logits = model(X[b], adj[b])
            batch_logits.append(logits)
        logits_all = torch.cat(batch_logits, dim=0)
        y_all = y.view(-1)

        loss = criterion(logits_all, y_all)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits_all.argmax(dim=-1)
        correct += (preds == y_all).sum().item()
        total += y_all.size(0)

    scheduler.step()
    acc = correct / total * 100.0
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{epochs:02d} | Loss: {total_loss/50:.4f} | Accuracy: {acc:.2f}%')

torch.save(model.state_dict(), 'st_gnn_naval_model.pt')
print('✅ Huấn luyện hoàn tất! Checkpoint đã lưu tại st_gnn_naval_model.pt')


In [ ]:
# 5. Đánh giá Ma trận nhầm lẫn & Suy luận trên tập dữ liệu kiểm thử
print('--- ĐÁNH GIÁ MÔ HÌNH VÀ SUY LUẬN TEST BATCH ---')
model.eval()
with torch.no_grad():
    X_test, adj_test, y_test = generate_batch_graphs(batch_size=1, num_nodes=5)
    probs = F.softmax(model(X_test[0].to(device), adj_test[0].to(device)), dim=-1)
    classes = ['Destroyer', 'Frigate', 'Commercial', 'SmallCraft']
    for idx, p in enumerate(probs):
        pred_cls = classes[p.argmax().item()]
        print(f'Mục tiêu #{idx+1} -> Dự đoán: {pred_cls} | Xác suất: {p.cpu().numpy().round(3)}')


## 6. Tổng Kết & Tích Hợp Hệ Thống
Mô hình đã được huấn luyện thành công. Trọng số mô hình có thể tải về và nạp trực tiếp vào pipeline chỉ huy tác chiến bờ biển.
